# NDR - Notebook 40: Trigger Reconstruction Probe (gated, no-submit)

## Goal
Answer one measured question, no guessing:
> Does a shared trigger reconstructed from 20 poison boxes fire the poisoned model much more than controls?

Probe only. No LB submission.

## Evidence already measured
| Source | Result | Meaning |
|---|---|---|
| NB01 transplant | 100/100 fire, conf 0.540 | Local trigger; real patch is oracle |
| NB07 free NC | fire 1/15=0.067 | Free-form reverse-engineer weak |
| NB30 dashed | dashed~continuous | Pure dashed synth insufficient |
| NB25/33/34 | 254/240/286 | Blurry filters/ensemble/multiscale dead |
| NB36-39 | forget ok, clean conf drift | Weight suppress does not beat 229 |
| arXiv 2605.05928 | class-oriented adv gen mismatches detector | Need localized detection objective |

## Why NB07 failed
1. Global top-k diluted signal
2. Zero init + fixed 40x40 mismatched poison sizes
3. No scramble/brightness controls
4. No oracle comparison

## Probe design
A oracle real patch
B mean/PCA shared template
C recon_da: init=mean, size=canonical, localized anchor loss
D controls: scramble + continuous

## Hard gates
FIRE_OK: shared>=0.60 and >=2x continuous
SPEC_OK: shared-scramble>=0.30
ORACLE_OK: shared/oracle>=0.50
USE_TRIGGER=all three


## 1. Setup


In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 6.0 MB/s eta 0:00:00


In [2]:
import copy, json, math, random, warnings, traceback, os, sys
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.engine import DefaultPredictor
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

print('python', sys.version)
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'ngpu', torch.cuda.device_count())

CANDIDATES = []
kaggle_input = Path('/kaggle/input')
if kaggle_input.exists():
    print('=== /kaggle/input listing ===')
    for p in sorted(kaggle_input.iterdir()):
        print(' ', p)
        CANDIDATES.append(p)
        try:
            for q in p.iterdir():
                if q.is_dir():
                    CANDIDATES.append(q)
        except Exception:
            pass
CANDIDATES.extend([
    Path('/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'),
    Path('/kaggle/input/neural-debris-removal-in-streak-detection-models'),
    Path(r'D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/data'),
])

def looks_like_root(c):
    if not c.exists() or not c.is_dir():
        return False
    w1 = c / 'poisoned_model' / 'poisoned_model.pth'
    w2 = c / 'poisoned_model.pth'
    u = c / 'unlearn_set'
    return (w1.exists() or w2.exists()) and u.exists()

ROOT = None
for c in CANDIDATES:
    try:
        if looks_like_root(c):
            ROOT = c
            break
    except Exception:
        pass
if ROOT is None and kaggle_input.exists():
    for w in kaggle_input.rglob('poisoned_model.pth'):
        cand = w.parent.parent if w.parent.name == 'poisoned_model' else w.parent
        if looks_like_root(cand):
            ROOT = cand
            break
if ROOT is None:
    raise FileNotFoundError('Cannot find competition data root. Attach competition data and rerun.')

if (ROOT / 'poisoned_model' / 'poisoned_model.pth').exists():
    POISONED_WEIGHTS = str(ROOT / 'poisoned_model' / 'poisoned_model.pth')
else:
    POISONED_WEIGHTS = str(ROOT / 'poisoned_model.pth')
UNLEARN_DIR = ROOT / 'unlearn_set'
TEST_DIR = ROOT / 'test_set' / 'test_set'
if not TEST_DIR.exists():
    TEST_DIR = ROOT / 'test_set'
if not TEST_DIR.exists():
    TEST_DIR = ROOT / 'test'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path(r'D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/_probe40')
WORK.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
CAND_THRESH = 0.05
FIRE_CONF = 0.20
MATCH_IOU = 0.10
MARGIN = 6
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('ROOT=', ROOT)
print('UNLEARN_DIR exists', UNLEARN_DIR.exists(), 'TEST_DIR exists', TEST_DIR.exists())
print('device=', DEVICE)
print('weights=', POISONED_WEIGHTS, 'exists', Path(POISONED_WEIGHTS).exists())


python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch 2.10.0+cu128 cuda True ngpu 2
=== /kaggle/input listing ===
  /kaggle/input/competitions
  /kaggle/input/datasets
ROOT= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models
UNLEARN_DIR exists True TEST_DIR exists True
device= cuda
weights= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth exists True


## 2. Loaders + poisoned model


In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    cfg.INPUT.FORMAT = 'RGB'
    return cfg

def build_predictor(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    return DefaultPredictor(build_cfg(weights, score_thresh))

def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    boxes = inst.pred_boxes.tensor.numpy().astype(np.float32)
    scores = inst.scores.numpy().astype(np.float32)
    return boxes, scores

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    aa = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    ab = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    union = aa + ab - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)

ann_path = UNLEARN_DIR / 'annotations_coco.json'
if not ann_path.exists():
    cands = list(UNLEARN_DIR.rglob('annotations_coco.json'))
    if not cands:
        raise FileNotFoundError('annotations_coco.json not under ' + str(UNLEARN_DIR))
    ann_path = cands[0]
    print('using nested annotations', ann_path)
with open(ann_path) as f:
    coco = json.load(f)
stem_by_id = {im['id']: str(Path(im['file_name']).stem) for im in coco['images']}

def stem_key(p):
    digs = ''.join([ch for ch in p.stem if ch.isdigit()])
    return int(digs) if digs else 0

unlearn_files = sorted(UNLEARN_DIR.glob('*.png'), key=stem_key)
if not unlearn_files:
    unlearn_files = sorted(UNLEARN_DIR.rglob('*.png'), key=stem_key)
test_files = sorted(TEST_DIR.glob('*.png'), key=stem_key)
if not test_files:
    test_files = sorted(TEST_DIR.rglob('*.png'), key=stem_key)
print('n unlearn png', len(unlearn_files), 'n test png', len(test_files), 'n coco images', len(coco.get('images', [])), 'n anns', len(coco.get('annotations', [])))
unlearn_imgs = {}
for p in unlearn_files:
    img = load_image(p)
    unlearn_imgs[str(p.stem)] = img
    unlearn_imgs[str(stem_key(p))] = img
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('predictor ready | unlearn_imgs keys sample', list(unlearn_imgs.keys())[:5])


n unlearn png 20 n test png 2000 n coco images 20 n anns 20


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


predictor ready | unlearn_imgs keys sample ['15', '104', '108', '147', '200']


## 3. Extract 20 real poison patches + size stats


In [4]:
poison_patches = []
missing = 0
for a in coco['annotations']:
    x, y, w, h = map(float, a['bbox'])
    stem = stem_by_id[a['image_id']]
    img = unlearn_imgs.get(str(stem))
    if img is None:
        digs = ''.join([ch for ch in str(stem) if ch.isdigit()])
        img = unlearn_imgs.get(str(int(digs))) if digs else None
    if img is None:
        missing += 1
        continue
    x0 = int(max(0, x - MARGIN)); y0 = int(max(0, y - MARGIN))
    x1 = int(min(IMG_W, x + w + MARGIN)); y1 = int(min(IMG_H, y + h + MARGIN))
    if x1 <= x0 + 2 or y1 <= y0 + 2:
        missing += 1
        continue
    patch = img[y0:y1, x0:x1].copy()
    poison_patches.append({
        'stem': stem,
        'bbox_xyxy': np.array([x, y, x + w, y + h], np.float32),
        'crop_xyxy': np.array([x0, y0, x1, y1], np.float32),
        'patch': patch,
        'inner_wh': (float(w), float(h)),
        'crop_wh': (patch.shape[1], patch.shape[0]),
    })
if len(poison_patches) == 0:
    raise RuntimeError('No poison patches extracted (missing=%d). Check unlearn stems vs coco.' % missing)
ws = np.array([p['inner_wh'][0] for p in poison_patches], np.float32)
hs = np.array([p['inner_wh'][1] for p in poison_patches], np.float32)
print('poison patches: %d missing=%d' % (len(poison_patches), missing))
print('inner w: med=%.1f p10=%.1f p90=%.1f' % (np.median(ws), np.percentile(ws,10), np.percentile(ws,90)))
print('inner h: med=%.1f p10=%.1f p90=%.1f' % (np.median(hs), np.percentile(hs,10), np.percentile(hs,90)))
CAN_W = int(np.clip(np.round(np.median([p['crop_wh'][0] for p in poison_patches])), 24, 96))
CAN_H = int(np.clip(np.round(np.median([p['crop_wh'][1] for p in poison_patches])), 24, 96))
if CAN_W % 2: CAN_W += 1
if CAN_H % 2: CAN_H += 1
print('canonical shared size HxW =', CAN_H, CAN_W)


poison patches: 20 missing=0
inner w: med=34.7 p10=22.3 p90=53.8
inner h: med=27.2 p10=10.5 p90=53.7
canonical shared size HxW = 40 46


## 4. Paste helpers + fire-at-box metric


In [5]:
def paste_patch(img, patch, x0, y0, feather=2):
    out = img.copy()
    ph, pw = patch.shape[:2]
    if y0 < 0 or x0 < 0 or y0 + ph > IMG_H or x0 + pw > IMG_W:
        return None
    reg = out[y0:y0+ph, x0:x0+pw]
    mask = np.ones((ph, pw, 1), np.float32)
    if feather > 0 and ph > 2 * feather and pw > 2 * feather:
        rh = np.minimum(np.arange(ph)[:, None] + 1, np.arange(ph)[::-1][:, None] + 1) / feather
        rw = np.minimum(np.arange(pw)[None, :] + 1, np.arange(pw)[::-1][None, :] + 1) / feather
        mask = np.clip(np.minimum(rh, rw), 0, 1)[..., None]
    out[y0:y0+ph, x0:x0+pw] = reg * (1 - mask) + patch.astype(np.float32) * mask
    return out

def fire_at_region(predictor, img, xyxy, conf=FIRE_CONF, iou_thr=MATCH_IOU):
    boxes, scores = predict_image(predictor, img)
    if len(boxes) == 0:
        return 0.0, 0.0
    gt = np.array([xyxy], np.float32)
    ious = iou_matrix(gt, boxes)[0]
    j = int(np.argmax(ious))
    if ious[j] >= iou_thr and scores[j] >= conf:
        return 1.0, float(scores[j])
    return 0.0, float(scores[j] if ious[j] >= iou_thr else 0.0)

def rand_free_spot(bg_img, pw, ph, rng, avoid=None, tries=40):
    for _ in range(tries):
        x0 = int(rng.integers(8, IMG_W - pw - 8))
        y0 = int(rng.integers(8, IMG_H - ph - 8))
        box = np.array([[x0, y0, x0 + pw, y0 + ph]], np.float32)
        if avoid is not None and len(avoid):
            if iou_matrix(box, avoid).max() >= 0.05:
                continue
        return x0, y0
    return None

def resize_patch(patch, pw, ph):
    return cv2.resize(patch.astype(np.float32), (pw, ph), interpolation=cv2.INTER_AREA)


## 5. Shared templates + controls


In [6]:
stack = np.stack([resize_patch(p['patch'], CAN_W, CAN_H) for p in poison_patches], 0)
mean_template = stack.mean(0).astype(np.float32)
gray = stack.mean(-1).reshape(len(stack), -1)
gray0 = gray - gray.mean(0, keepdims=True)
U, S, Vt = np.linalg.svd(gray0, full_matrices=False)
comp = Vt[0].reshape(CAN_H, CAN_W)
comp = comp / (np.abs(comp).max() + 1e-6)
pca_template = mean_template.copy()
for c in range(3):
    pca_template[:, :, c] = np.clip(mean_template[:, :, c] + 8.0 * comp, 0, 255)

def scramble_template(tmpl, rng):
    flat = tmpl.reshape(-1, 3).copy()
    rng.shuffle(flat)
    return flat.reshape(tmpl.shape).astype(np.float32)

def continuous_streak_patch(pw, ph, rng, bright_scale=None):
    canvas = np.zeros((ph, pw), np.float32)
    ang = rng.uniform(0, np.pi)
    L = 0.9 * min(pw, ph)
    cx, cy = pw / 2, ph / 2
    p1 = (int(cx - L/2 * np.cos(ang)), int(cy - L/2 * np.sin(ang)))
    p2 = (int(cx + L/2 * np.cos(ang)), int(cy + L/2 * np.sin(ang)))
    cv2.line(canvas, p1, p2, 1.0, max(1, int(round(min(pw, ph) * 0.08))))
    canvas = cv2.GaussianBlur(canvas, (0, 0), 0.8)
    if bright_scale is None:
        bright_scale = float(np.percentile(mean_template, 90) - np.percentile(mean_template, 20))
    bg = float(np.percentile(mean_template, 20))
    g = bg + canvas / (canvas.max() + 1e-6) * bright_scale
    g = np.clip(g, 0, 255).astype(np.float32)
    return np.repeat(g[:, :, None], 3, axis=2)

rng = np.random.default_rng(SEED)
scramble_tmpl = scramble_template(mean_template, rng)
print('templates ready | mean range', float(mean_template.min()), float(mean_template.max()))


templates ready | mean range 37.47188186645508 118.4183578491211


## 6. Detection-aware reverse-engineer (fix NB07)


In [7]:
def _norm_img(model, x255):
    mean = model.pixel_mean
    std = model.pixel_std
    return (x255 - mean) / std

def region_anchor_scores(model, x255, y0, x0, ph, pw):
    # Detection-localized scores only inside the patch region.
    nrm = _norm_img(model, x255)[None]
    feats = model.backbone(nrm)
    feat_list = [feats[f] for f in model.head_in_features]
    out = model.head(feat_list)
    logits_list = out[0] if isinstance(out, (tuple, list)) else out
    scores = []
    for logit in logits_list:
        if not torch.is_tensor(logit) or logit.dim() != 4:
            continue
        _, C, Hi, Wi = logit.shape
        stride_y = IMG_H / float(Hi)
        stride_x = IMG_W / float(Wi)
        ys = (torch.arange(Hi, device=logit.device, dtype=torch.float32) + 0.5) * stride_y
        xs = (torch.arange(Wi, device=logit.device, dtype=torch.float32) + 0.5) * stride_x
        yy = ys.view(Hi, 1).expand(Hi, Wi)
        xx = xs.view(1, Wi).expand(Hi, Wi)
        inside = (yy >= float(y0)) & (yy < float(y0 + ph)) & (xx >= float(x0)) & (xx < float(x0 + pw))
        if int(inside.sum().item()) == 0:
            continue
        prob = torch.sigmoid(logit[0])
        m = prob.max(dim=0).values
        scores.append(m[inside])
    if not scores:
        allp = []
        for logit in logits_list:
            if torch.is_tensor(logit):
                allp.append(torch.sigmoid(logit).flatten())
        if not allp:
            return torch.zeros(1, device=x255.device)
        return torch.cat(allp)
    return torch.cat(scores)

def reverse_trigger_da(model, init_tmpl, n_bg=2, iters=40, lr=0.05, lam=5e-4, seed=SEED):
    # Memory-safe version:
    # - keep backgrounds on CPU
    # - sample 1 image / step (not 16 simultaneous full forwards)
    # - base forward under no_grad
    # - free graph each step
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    rng = np.random.default_rng(seed)

    # pick a few low-det backgrounds once (CPU only)
    n_pick = min(20, len(test_files))
    idxs = rng.choice(len(test_files), size=n_pick, replace=False)
    scored = []
    for k in idxs:
        img = load_image(test_files[int(k)])
        with torch.no_grad():
            b, s = predict_image(poisoned_predictor, img)
        scored.append((img, int((s >= 0.2).sum()) if len(s) else 0))
    bgs_cpu = [im for im, _ in sorted(scored, key=lambda t: t[1])[:max(2, n_bg)]]

    ph, pw = init_tmpl.shape[:2]
    init = torch.as_tensor(init_tmpl.transpose(2, 0, 1).copy(), dtype=torch.float32, device=DEVICE)
    delta = init.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([delta], lr=lr)

    for it in range(iters):
        opt.zero_grad(set_to_none=True)
        # one random background per step to avoid OOM
        img = bgs_cpu[int(rng.integers(len(bgs_cpu)))]
        x = torch.as_tensor(img.transpose(2, 0, 1).copy(), dtype=torch.float32, device=DEVICE)
        y0 = int(rng.integers(0, IMG_H - ph))
        x0 = int(rng.integers(0, IMG_W - pw))

        with torch.no_grad():
            base = region_anchor_scores(model, x, y0, x0, ph, pw).mean()

        xt = x.clone()
        d = torch.clamp(delta, 0, 255)
        xt[:, y0:y0+ph, x0:x0+pw] = d
        withd = region_anchor_scores(model, xt, y0, x0, ph, pw).mean()
        loss = -(withd - base) + lam * (delta - init).pow(2).mean()
        loss.backward()
        opt.step()
        with torch.no_grad():
            delta.clamp_(0, 255)

        # free activations
        del x, xt, withd, base, loss
        if (it + 1) % 10 == 0 or it == 0:
            print('  recon_da it%d' % (it + 1))
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()

    out = torch.clamp(delta.detach(), 0, 255).cpu().numpy().transpose(1, 2, 0).astype(np.float32)
    return out

print('reverse_trigger_da defined (memory-safe)')


reverse_trigger_da defined (memory-safe)


In [8]:
print('Starting recon_da (memory-safe) ...')
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
try:
    rev_model = build_model(build_cfg(POISONED_WEIGHTS))
    DetectionCheckpointer(rev_model).load(POISONED_WEIGHTS)
    rev_model.to(DEVICE)
    RECON_DA = reverse_trigger_da(
        rev_model,
        init_tmpl=mean_template,
        n_bg=2,
        iters=30 if DEVICE == 'cpu' else 40,
        lr=0.05,
        lam=5e-4,
    )
    del rev_model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print('RECON_DA ready', RECON_DA.shape, 'range', float(RECON_DA.min()), float(RECON_DA.max()))
    np.save(WORK / 'recon_da.npy', RECON_DA)
    np.save(WORK / 'mean_template.npy', mean_template)
    print('saved templates to', WORK)
except Exception as e:
    print('RECON FAILED:', repr(e))
    traceback.print_exc()
    RECON_DA = mean_template.copy()
    np.save(WORK / 'recon_da_fallback_mean.npy', RECON_DA)
    print('Using mean_template as RECON_DA fallback so probe can continue')
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


Starting recon_da (memory-safe) ...

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.



  recon_da it1
  recon_da it10
  recon_da it20
  recon_da it30
  recon_da it40
RECON_DA ready (40, 46, 3) range 36.50193786621094 118.30459594726562
saved templates to /kaggle/working


## 7. Fire-rate evaluation vs controls


In [9]:
def eval_variant(name, patch_builder, n=40, seed=0):
    rng = np.random.default_rng(seed)
    bgs = []
    seen = set()
    for v in unlearn_imgs.values():
        vid = id(v)
        if vid in seen:
            continue
        seen.add(vid)
        bgs.append(v)
    n_extra = min(10, len(test_files))
    if n_extra > 0:
        extra_idx = rng.choice(len(test_files), size=n_extra, replace=False)
        extra = [load_image(test_files[int(i)]) for i in extra_idx]
        bgs = bgs + extra
    fires, confs = [], []
    for i in range(n):
        patch = patch_builder(rng)
        ph, pw = patch.shape[:2]
        if ph < 8 or pw < 8:
            continue
        bg = bgs[int(rng.integers(len(bgs)))]
        spot = rand_free_spot(bg, pw, ph, rng)
        if spot is None:
            continue
        x0, y0 = spot
        im2 = paste_patch(bg, patch, x0, y0)
        if im2 is None:
            continue
        xyxy = [x0 + 2, y0 + 2, x0 + pw - 2, y0 + ph - 2]
        f, c = fire_at_region(poisoned_predictor, im2, xyxy)
        fires.append(f); confs.append(c)
    fires = np.array(fires, np.float32) if fires else np.zeros(0, np.float32)
    confs = np.array(confs, np.float32) if confs else np.zeros(0, np.float32)
    return {
        'name': name,
        'n': int(len(fires)),
        'fire_rate': float(fires.mean()) if len(fires) else 0.0,
        'mean_conf_when_scored': float(confs.mean()) if len(confs) else 0.0,
        'mean_conf_when_fired': float(confs[fires > 0].mean()) if (fires > 0).any() else 0.0,
    }

N_EVAL = 20 if DEVICE == 'cpu' else 30

def builder_oracle(rng):
    return poison_patches[int(rng.integers(len(poison_patches)))]['patch']

def builder_mean(rng):
    return mean_template.copy()

def builder_pca(rng):
    return pca_template.copy()

def builder_scramble(rng):
    return scramble_template(mean_template, rng)

def builder_cont(rng):
    return continuous_streak_patch(CAN_W, CAN_H, rng)

def builder_recon(rng):
    return RECON_DA.copy()

variants = [
    ('oracle_real', builder_oracle),
    ('mean_template', builder_mean),
    ('pca_template', builder_pca),
    ('scramble', builder_scramble),
    ('continuous', builder_cont),
    ('recon_da', builder_recon),
]

rows = []
for name, b in variants:
    print('eval', name, '...')
    seed = SEED + (sum(ord(ch) for ch in name) % 1000)
    rows.append(eval_variant(name, b, n=N_EVAL, seed=seed))

df = pd.DataFrame(rows).sort_values('fire_rate', ascending=False)
print('=== FIRE TABLE ===')
print(df.to_string(index=False))
df.to_csv(WORK / 'probe40_fire_table.csv', index=False)
print('saved', WORK / 'probe40_fire_table.csv')


eval oracle_real ...
eval mean_template ...
eval pca_template ...
eval scramble ...
eval continuous ...
eval recon_da ...
=== FIRE TABLE ===
         name  n  fire_rate  mean_conf_when_scored  mean_conf_when_fired
   continuous 30   0.900000               0.493322              0.529472
  oracle_real 30   0.666667               0.439887              0.600723
 pca_template 30   0.566667               0.285174              0.438087
     recon_da 30   0.533333               0.310306              0.501223
mean_template 30   0.466667               0.249822              0.428158
     scramble 30   0.000000               0.011322              0.000000
saved /kaggle/working/probe40_fire_table.csv


## 8. Gate decision (no guessing)


In [10]:
def get_rate(name):
    r = df.loc[df['name'] == name, 'fire_rate']
    return float(r.iloc[0]) if len(r) else 0.0

oracle_fire = get_rate('oracle_real')
recon_fire = get_rate('recon_da')
mean_fire = get_rate('mean_template')
pca_fire = get_rate('pca_template')
scramble_fire = get_rate('scramble')
cont_fire = get_rate('continuous')

shared_name, shared_fire = max(
    [('mean_template', mean_fire), ('pca_template', pca_fire), ('recon_da', recon_fire)],
    key=lambda t: t[1]
)

FIRE_OK = (shared_fire >= 0.60) and (shared_fire >= 2.0 * max(cont_fire, 1e-6))
SPEC_OK = (shared_fire - scramble_fire) >= 0.30
ORACLE_OK = (shared_fire / max(oracle_fire, 1e-6)) >= 0.50
USE_TRIGGER = bool(FIRE_OK and SPEC_OK and ORACLE_OK)

print('=' * 64)
print('NB40 TRIGGER RECON PROBE - GATES')
print('=' * 64)
print(f'oracle_real fire     = {oracle_fire:.3f}  (NB01 expected ~1.0)')
print(f'mean_template fire   = {mean_fire:.3f}')
print(f'pca_template fire    = {pca_fire:.3f}')
print(f'recon_da fire        = {recon_fire:.3f}')
print(f'scramble control     = {scramble_fire:.3f}')
print(f'continuous control   = {cont_fire:.3f}')
print(f'best shared          = {shared_name} @ {shared_fire:.3f}')
print('-' * 64)
print(f'FIRE_OK   = {FIRE_OK}   # shared>=0.60 and >=2x continuous')
print(f'SPEC_OK   = {SPEC_OK}   # shared-scramble >= 0.30')
print(f'ORACLE_OK = {ORACLE_OK} # shared/oracle >= 0.50')
print(f'USE_TRIGGER = {USE_TRIGGER}')
print('-' * 64)
if USE_TRIGGER:
    print('PASS: shared trigger signal is real.')
    print('Next legal step: NB41 inference scorer/filter from best shared trigger.')
    print('Do NOT train suppress head (avoid NB36-39 drift).')
    print('A/B vs base229 only; submit only if beats 229 on LB.')
else:
    print('FAIL: shared-trigger reconstruction not strong/specific enough.')
    print('Logical conclusion from current evidence:')
    print('- Real poison patches remain local triggers if oracle_fire is high.')
    print('- But no shared delta reconstructs strongly beyond controls.')
    print('- Hypothesis "145 = Neural Cleanse shared trigger" is weakened here.')
    print('- Keep 229 as neo; do not chase NC-shared further without new measured idea.')
print('=' * 64)

decision = {
    'oracle_fire': oracle_fire,
    'mean_fire': mean_fire,
    'pca_fire': pca_fire,
    'recon_fire': recon_fire,
    'scramble_fire': scramble_fire,
    'cont_fire': cont_fire,
    'best_shared': shared_name,
    'shared_fire': shared_fire,
    'FIRE_OK': FIRE_OK,
    'SPEC_OK': SPEC_OK,
    'ORACLE_OK': ORACLE_OK,
    'USE_TRIGGER': USE_TRIGGER,
}
with open(WORK / 'probe40_decision.json', 'w') as f:
    json.dump(decision, f, indent=2)
print('saved', WORK / 'probe40_decision.json')


NB40 TRIGGER RECON PROBE - GATES
oracle_real fire     = 0.667  (NB01 expected ~1.0)
mean_template fire   = 0.467
pca_template fire    = 0.567
recon_da fire        = 0.533
scramble control     = 0.000
continuous control   = 0.900
best shared          = pca_template @ 0.567
----------------------------------------------------------------
FIRE_OK   = False   # shared>=0.60 and >=2x continuous
SPEC_OK   = True   # shared-scramble >= 0.30
ORACLE_OK = True # shared/oracle >= 0.50
USE_TRIGGER = False
----------------------------------------------------------------
FAIL: shared-trigger reconstruction not strong/specific enough.
Logical conclusion from current evidence:
- Real poison patches remain local triggers if oracle_fire is high.
- But no shared delta reconstructs strongly beyond controls.
- Hypothesis "145 = Neural Cleanse shared trigger" is weakened here.
- Keep 229 as neo; do not chase NC-shared further without new measured idea.
saved /kaggle/working/probe40_decision.json


## 9. Optional diagnostic: template specificity


In [11]:
sims = []
for p in poison_patches:
    r = resize_patch(p['patch'], CAN_W, CAN_H)
    a = (r.mean(-1) - r.mean()).ravel()
    b = (mean_template.mean(-1) - mean_template.mean()).ravel()
    cos = float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-6) * (np.linalg.norm(b) + 1e-6)))
    sims.append(cos)
sims = np.array(sims, np.float32)
cont_sims = []
rng = np.random.default_rng(123)
for _ in range(20):
    r = continuous_streak_patch(CAN_W, CAN_H, rng)
    a = (r.mean(-1) - r.mean()).ravel()
    b = (mean_template.mean(-1) - mean_template.mean()).ravel()
    cos = float(np.dot(a, b) / ((np.linalg.norm(a) + 1e-6) * (np.linalg.norm(b) + 1e-6)))
    cont_sims.append(cos)
cont_sims = np.array(cont_sims, np.float32)
print(f'cos(real_poison, mean_tmpl): med={np.median(sims):.3f} mean={sims.mean():.3f}')
print(f'cos(continuous, mean_tmpl):  med={np.median(cont_sims):.3f} mean={cont_sims.mean():.3f}')
print('If distributions heavily overlap, shared template is not specific (explains gate fail).')


cos(real_poison, mean_tmpl): med=0.337 mean=0.341
cos(continuous, mean_tmpl):  med=0.432 mean=0.366
If distributions heavily overlap, shared template is not specific (explains gate fail).


## 10. Allowed conclusions

| Result | Valid conclusion | Next |
|---|---|---|
| USE_TRIGGER=True | Shared trigger recon has real signal | Build NB41 scorer/filter, A/B vs 229 |
| USE_TRIGGER=False, oracle high | Local trigger exists, but not shared | 145 not explained by shared NC here; keep 229 |
| oracle low | Load/path/model bug | Fix load and rerun; do not conclude |

Forbidden: submit from NB40; train repair in NB40; loosen gates to force pass.
